Description: Get peak information, perform coincidence, and plot summed spectrum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import glob
import os
sys.path.insert(0,"/home/ws/sk6801/sw/UCSD_analysis/sandpro")
import sandpro
import configparser
import json
import scipy.stats
from matplotlib.colors import LogNorm

from scipy.optimize import curve_fit
import datetime
import pandas as pd
from copy import deepcopy
from numba import jit
import time

sys.path.insert(0,"../src/")
import common.d2d as d2d
import common.utils as util
import data_processing.fast_processor_all_channel as fast_processor
from data_processing.event_processor_all_channel import EventProcessor
from application.peak_processor import Peak_Processor

from data_structure.waveform_info import WaveformInfo
from data_structure.peak_info import PeakInfo

%run "plot_style_kalinka.py"


### Import and Process Data

In [ ]:
df_SPE_position = pd.read_csv(
    "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spe_position_LXe_2.csv",
                 delimiter=",")

df_SPE_position

In [ ]:
peak_processor = Peak_Processor()

df = pd.read_csv(
    # "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250706_LXe_gain_info_single_channel.csv",
    "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250616_LXe_gain_info_single_channel.csv", 
                 parse_dates=["date_time"],
                 delimiter=",",
                 quotechar='"', 
                 skipinitialspace=True, 
                 encoding="utf-8")


# convert string to array
if isinstance(df['board_0_channels'][0], str):
    if "," in df['board_0_channels'][0]:
        df['board_0_channels'] = df['board_0_channels'].apply(json.loads).apply(np.array)
    else:
        df['board_0_channels'] = df['board_0_channels'].apply(lambda x: x.replace("  ", ","))
        df['board_0_channels'] = df['board_0_channels'].apply(lambda x: x.replace("[ ", "["))
        df['board_0_channels'] = df['board_0_channels'].apply(lambda x: x.replace(" ", ","))
        df['board_0_channels'] = df['board_0_channels'].apply(json.loads).apply(np.array)

        df['board_1_channels'] = df['board_1_channels'].apply(lambda x: x.replace("  ", ","))
        df['board_1_channels'] = df['board_1_channels'].apply(lambda x: x.replace("[ ", "["))
        df['board_1_channels'] = df['board_1_channels'].apply(lambda x: x.replace(" ", ","))
        df['board_1_channels'] = df['board_1_channels'].apply(json.loads).apply(np.array)

print(len(df))
idx_nan = np.where((df['spe_position'].isna()) & (df['voltage_preamp1_V']<-46))[0]
# replace NaN values with SPE position
voltage_array = df.loc[idx_nan, 'voltage_preamp1_V']
channel_array = df.loc[idx_nan, 'channel']
print(len(idx_nan))
for idx, voltage, channel in zip(idx_nan, voltage_array, channel_array):
    # find the SPE position for the given voltage and channel
    df.loc[idx, 'spe_position'] = df_SPE_position[
        (df_SPE_position['voltage_preamp1_V'] == voltage) & 
        (df_SPE_position['channel'] == channel)
    ]['spe_position'].values
    df.loc[idx, 'spe_position_err'] = df_SPE_position[
        (df_SPE_position['voltage_preamp1_V'] == voltage) & 
        (df_SPE_position['channel'] == channel)
    ]['spe_position_err'].values


In [ ]:
#### Basic Data Selection

all_runs_d2d = d2d.data(df)


mask_data_taking_mode = (all_runs_d2d.data_taking_mode == "all_channels")
mask_na = ~np.isnan(all_runs_d2d.spe_position)
# mask = ~np.isnan(all_runs_d2d.gain)
# mask_voltage = (all_runs_d2d.voltage_preamp1_V == -49)
# mask_run_tag = (all_runs_d2d.run_tag == "LXe/Cs137")
mask_run_tag = (all_runs_d2d.run_tag == "LXe/gain_calibration")
mask = mask_data_taking_mode & mask_na & mask_run_tag
# & mask_voltage

all_runs_d2d.apply_mask(mask, inplace=True, dry = False)
all_run_list = np.unique(all_runs_d2d.md_full_path)

#### Run Selection

check how many runs have all 24 channel data, print a list

In [ ]:
count_successful = 0
count_failed = 0

count_n_events = 0

for i, md_full_path in enumerate(all_run_list):

    mask = all_runs_d2d.md_full_path == md_full_path
    single_run = all_runs_d2d.apply_mask(mask, inplace=False, dry = True)

    if len(single_run.channel) < 24:
        # print(f"Run {md_full_path} has {len(single_run.channel)} channels, expected 24 channels. Run tag: {single_run.run_tag[0]}. Comment: {single_run.comment[0]}. Voltage: {single_run.voltage_preamp1_V[0]}")
        count_failed += 1

        continue    
    elif len(single_run.channel) == 24:
        print(f"Run {i}: {md_full_path} has 24 channels. Run tag: {single_run.run_tag[0]}. Voltage: {single_run.voltage_preamp1_V[0]}. Comment: {single_run.comment[0]}. Event number: {single_run.n_processed_events[0]}")
        count_successful += 1
        count_n_events += single_run.n_processed_events[0]
        
    else: 
        raise ValueError(f"Run {md_full_path} has {len(single_run.channel)} channels, expected 24 channels.")

print(f"Total runs: {len(all_run_list)}"
      f"Successful runs: {count_successful} "
      f"Failed runs: {count_failed} "
      f"Success rate: {count_successful/len(all_run_list)*100:.2f}%")

print(f"Total number of events: {count_n_events}")

#### Process Run

In [ ]:
### Choose a run from the list above
# initialize the result storage
df_result = pd.DataFrame(columns=WaveformInfo().__dict__.keys())
single_info_list = []

run_id=68
for md_full_path in all_run_list[run_id:run_id+1]:
# for md_full_path in all_run_list:
    result, waveform, baseline, baseline_std = peak_processor.get_peak_level_data(
        all_runs_d2d=all_runs_d2d,
        md_full_path=md_full_path,
        peak_merge_window_sample=0
    )
    single_info_list += result

df_result = pd.DataFrame.from_dict(single_info_list)
d2d_data = d2d.data(df_result)



### Check Results (Plots)

In [ ]:
# data selection
# mask = (d2d_data.peak_area_PE > 1.5) 
# mask = (d2d_data.peak_height_V < 1.2) 
# mask = (d2d_data.peak_width_ns < 300) & (d2d_data.peak_width_ns < 750) & (d2d_data.peak_width_ns > 500)
# mask = (d2d_data.peak_width_ns > 300)
# selected_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
selected_data = d2d_data

plt.close()
fig_peak_integral_area, axes_peak_integral_area = plt.subplots(3,8,figsize=(35,15))
fig_area_height, axes_area_height = plt.subplots(3,8,figsize=(35,15))
fig_area_width, axes_area_width = plt.subplots(3,8,figsize=(35,15))
fig_area_height_full, axes_area_height_full = plt.subplots(3,8,figsize=(35,15))
fig_area_width_full, axes_area_width_full = plt.subplots(3,8,figsize=(35,15), sharex=True, sharey='col')
fig_width_height, axes_width_height = plt.subplots(3,8,figsize=(35,15))

axes = [axes_area_height, axes_area_height_full, axes_area_width, axes_area_width_full, axes_width_height, axes_peak_integral_area]
figures = [fig_area_height, fig_area_height_full, fig_area_width, fig_area_width_full, fig_width_height, fig_peak_integral_area] 
axes_name = ['axes_area_height', 
            'axes_area_height_full', 
            'axes_area_width', 
            'axes_area_width_full', 
            'axes_width_height', 
            'axes_peak_integral_area']



for channel in range(24):
    mask = selected_data.channel == channel
    singl_channel_data = selected_data.apply_mask(mask, inplace=False, dry=True)

    # # this also remove null values from the peak_height_V_array
    # array = list(singl_channel_data.peak_height_V_array)
    # height_V = np.concatenate(array)
    # array = list(singl_channel_data.peak_area_PE_array)
    # area_Vns = np.concatenate(array)
    height_V = singl_channel_data.peak_height_V
    area_PE = singl_channel_data.peak_area_PE
    width_ns = singl_channel_data.peak_width_ns
    integral_window_area_PE = singl_channel_data.integral_window_area_PE
    # integral_window_area_PE = list(singl_channel_data.integral_window_area_PE)
    # integral_window_area_PE = np.concatenate(integral_window_area_PE)

    plot_row = channel % 3
    plot_col = channel // 3

    # change the rows so that it match with the physical layout of the channels
    if plot_row == 0:
        plot_row = 1
    elif plot_row == 1:
        plot_row = 0

    axes_peak_integral_area[plot_row,plot_col].hist((area_PE),
                            bins=100,
                            range=[-0.1,10], 
                            alpha=0.5,
                            label='peak_area_PE_array')
    axes_peak_integral_area[plot_row,plot_col].hist((integral_window_area_PE),
                            bins=100,
                            range=[-0.1,10], 
                            alpha=0.5,
                            label='integral_window_area_PE')   
    axes_peak_integral_area[plot_row,plot_col].set_yscale('log')
    if plot_row == 2:
        axes_peak_integral_area[plot_row,plot_col].set(xlabel='Area [PE]')
    if plot_col == 0:
        axes_peak_integral_area[plot_row,plot_col].set(ylabel='Counts')

    axes_area_height[plot_row,plot_col].hist2d(area_PE,height_V,
                            bins=[100,100],
                            range=[[-0.1,10],[0,0.1]],
                            cmap='viridis',
                            norm=LogNorm())
    if plot_row == 2:
        axes_area_height[plot_row,plot_col].set(xlabel='Area [PE]')
    if plot_col == 0:
        axes_area_height[plot_row,plot_col].set(ylabel='Height [V]')


    axes_area_width[plot_row,plot_col].hist2d(area_PE,width_ns,
                            bins=[100,100],
                            range=[[-0.1,10],[0,1000]],
                            cmap='viridis',
                            norm=LogNorm())  
    if plot_row == 2:
        axes_area_width[plot_row,plot_col].set(xlabel='Area [PE]')
    if plot_col == 0:
        axes_area_width[plot_row,plot_col].set(ylabel='Width [ns]')

    
    axes_area_height_full[plot_row,plot_col].hist2d(area_PE,height_V,
                            bins=[100,100],
                            range=[[-0.1,1500],[0,1.5]],
                            cmap='viridis',
                            norm=LogNorm())
    if plot_row == 2:
        axes_area_height_full[plot_row,plot_col].set(xlabel='Area [PE]')
    if plot_col == 0:
        axes_area_height_full[plot_row,plot_col].set(ylabel='Height [V]')


    axes_area_width_full[plot_row,plot_col].hist2d(area_PE,width_ns,
                            bins=[50,50],
                            range=[[-0.1,1000],[0,2500]],
                            cmap='viridis',
                            norm=LogNorm())  
    if plot_row == 2:
        axes_area_width_full[plot_row,plot_col].set(xlabel='Area [PE]')
    if plot_col == 0:
        axes_area_width_full[plot_row,plot_col].set(ylabel='Width [ns]')
    
    
    axes_width_height[plot_row,plot_col].hist2d(width_ns,height_V,
                            bins=[50,50],
                            range=[[0,1000],[0,1.5]],
                            cmap='viridis',
                            norm=LogNorm())  
    if plot_row == 2:
        axes_width_height[plot_row,plot_col].set(xlabel='Width [ns]')
    if plot_col == 0:
        axes_width_height[plot_row,plot_col].set(ylabel='Height [V]')
    
    for ax in axes:
        ax[plot_row,plot_col].set_title(f"Channel {channel}")

# Move title upwards
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.legend(axes_peak_integral_area, loc='upper right', fontsize='small')
axes_peak_integral_area[2,7].legend(loc="upper right")

# Set plot title
for fig, ax_name in zip(figures, axes_name):
    short_name = selected_data.md_full_path[0][0].replace('/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/','')
    short_name = short_name.replace('/','_')

    fig.suptitle(f"{ax_name}: \nPath: {short_name}\nComment: {selected_data.comment[0]}\nRun tag: {selected_data.run_tag[0]}\nVoltage: {selected_data.voltage_preamp1_V[0]}",
                 y=1.02, x=0.1,ha='left')
    
    
    # os.makedirs(short_name, exist_ok=True)
    # fig.savefig(f"{short_name}/{ax_name}.png", dpi=300, bbox_inches='tight')

In [ ]:
# data selection
# mask = (d2d_data.peak_area_PE > 1.5) 
# mask = (d2d_data.peak_height_V < 1.2) 
# mask = (d2d_data.peak_width_ns < 300) & (d2d_data.peak_width_ns < 750) & (d2d_data.peak_width_ns > 500)
# mask = (d2d_data.peak_width_ns > 300)
# selected_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
selected_data = d2d_data

plt.close()
fig_area_height_full, axes_area_height_full = plt.subplots(3,8,figsize=(15,6), sharex=True, sharey=True)
# gap between subplots
fig_area_height_full.subplots_adjust(wspace=0)
fig_area_height_full.subplots_adjust(hspace=0)

# axes = [axes_area_height_full]
# figures = [fig_area_height_full] 
# axes_name = [
#             'axes_area_height_full', 
#             ]

for channel in range(24):
    mask = selected_data.channel == channel
    singl_channel_data = selected_data.apply_mask(mask, inplace=False, dry=True)

    # # this also remove null values from the peak_height_V_array
    # array = list(singl_channel_data.peak_height_V_array)
    # height_V = np.concatenate(array)
    # array = list(singl_channel_data.peak_area_PE_array)
    # area_Vns = np.concatenate(array)
    height_V = singl_channel_data.peak_height_V
    area_PE = singl_channel_data.peak_area_PE
    width_ns = singl_channel_data.peak_width_ns
    integral_window_area_PE = singl_channel_data.integral_window_area_PE
    # integral_window_area_PE = list(singl_channel_data.integral_window_area_PE)
    # integral_window_area_PE = np.concatenate(integral_window_area_PE)

    plot_row = channel % 3
    plot_col = channel // 3

    # change the rows so that it match with the physical layout of the channels
    if plot_row == 0:
        plot_row = 1
    elif plot_row == 1:
        plot_row = 0

    axes_area_height_full[plot_row,plot_col].hist2d(area_PE,height_V,
                            bins=[100,100],
                            range=[[-0.1,1500],[0,1.5]],
                            cmap='viridis',
                            norm=LogNorm())
    # if plot_row == 2:
    #     axes_area_height_full[plot_row,plot_col].set(xlabel='Area [PE]')
    # if plot_col == 0:
    #     axes_area_height_full[plot_row,plot_col].set(ylabel='Height [V]')

    axes_area_height_full[plot_row,plot_col].text(0.4, 0.4, f"Ch. {channel}", transform=axes_area_height_full[plot_row,plot_col].transAxes, ha='left')

    axes_area_height_full[plot_row,plot_col].set_xlim([-46, 1500])
    axes_area_height_full[plot_row,plot_col].set_ylim([-0.2, 1.6])
# # Move title upwards
# plt.tight_layout(rect=[0, 0.03, 1, 0.95])
# # plt.legend(axes_peak_integral_area, loc='upper right', fontsize='small')
# # axes_peak_integral_area[2,7].legend(loc="upper right")

fig_area_height_full.supxlabel('Area [PE]', y=0.01)  # with adjusted position
fig_area_height_full.supylabel('Height [V]', x=0.09)  # with adjusted position


# color bar
cbar = fig_area_height_full.colorbar(axes_area_height_full[0, 0].collections[0], ax=axes_area_height_full, orientation='vertical', fraction=0.02, pad=0.04)
cbar.set_label('Counts', rotation=270, labelpad=5)

plt.savefig(f"/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/saturation_per_channel.png", dpi=100, bbox_inches='tight')

In [ ]:
plt.close()
fig, axes = plt.subplots(3,8,figsize=(35,15))

# masking
for channel in range(24):
    mask = d2d_data.channel == channel
    singl_channel_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

    # # this also remove null values from the peak_height_V_array
    # array = list(singl_channel_data.peak_height_V_array)
    # height_V = np.concatenate(array)
    # array = list(singl_channel_data.peak_area_PE_array)
    # area_Vns = np.concatenate(array)
    height_V = singl_channel_data.peak_height_V

    plot_row = channel % 3
    plot_col = channel // 3

    # change the rows so that it match with the physical layout of the channels
    if plot_row == 0:
        plot_row = 1
    elif plot_row == 1:
        plot_row = 0

    axes[plot_row,plot_col].hist(height_V,
                            bins=100,
                            range=[0,0.1])
    
    axes[plot_row,plot_col].set_title(f"Channel {channel}")

# set the labels
for ax in axes.flat:
    ax.set(xlabel='Height [V]', ylabel='Count')
    # log y
    ax.set_yscale('log')
    # ax.set_xlim(-0.1, 100)

# Set plot title
# plt.suptitle(f"Dataset: {path.split('/')[-1]}")

# Move title upwards
plt.tight_layout(rect=[0, 0.03, 1, 0.95])


plt.show()

### Area vs Time

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
# this also remove null values from the peak_height_V_array
peak_rel_start_time = d2d_data.peak_rel_start_time_s
peak_area = d2d_data.peak_area_PE
ax.set_xlabel('Time [$\mu$s]')

log = True
if not log:
        ax.hist2d(peak_rel_start_time*1e6, (peak_area),
                bins=[100,100],
                range=[[0.1,4],[-1,10]],
                cmap='viridis',
                norm=LogNorm()
                ) 
        # ax.axhline(1)
        ax.set_ylabel('Area [PE]')

else: 
        ax.hist2d(peak_rel_start_time*1e6, np.log10(peak_area),
                bins=[100,100],
                range=[[0,4],[-1,3]],
                cmap='viridis',
                norm=LogNorm()
                ) 
        # ax.axhline(0)
        ax.set_ylabel('Pulse area [PE]')

# configure the y axis labels and ticks
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(['1', '$10^{1}$', '$10^{2}$', '$10^{3}$'])

# color bar
cbar = plt.colorbar(ax.collections[0], ax=ax, orientation='vertical')
cbar.set_label('Counts')

# grid = True
ax.grid(True, color='black')

# plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/area_time_nosource.pdf")

#### Afterpulse fit

In [ ]:
df_result['first_peak_start_time'] = df_result.groupby('event_id')['peak_rel_start_time_s'].transform('min')
df_result['time_diff'] = df_result['peak_rel_start_time_s'] - df_result['first_peak_start_time']


fig, ax = plt.subplots(figsize=(12,5))

# peak_rel_start_time = d2d_data.peak_rel_start_time_s
time_diff = df_result['time_diff']

d2d_data.get_df()

ax.set_xlabel('Time$_{peak, i}$ - Time$_{first\\text{ }peak}$ [$\mu$s]')

ax.hist(time_diff*1e6, 
        bins=100,
        range=[0,4],
        alpha = 0.7,
         label='data from a single dataset'
        ) 
ax.set_ylabel('Count')

# log y
ax.set_yscale('log')

hist, bin_edges = np.histogram(time_diff*1e6, bins=100, range=[0,4])
bin_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])
hist_log = np.log10(hist + 1)  # add 1 to avoid log(0)
# ax.plot(bin_centers, hist_log, 'b-', label='Data')

# fit the decay trend
mask = (bin_centers >= 1) & (bin_centers <= 2.5)
# linear fit
p, cov = np.polyfit(bin_centers[mask], hist_log[mask], deg=1, cov=True)


# find tau
tau = -1 / p[0]
a = np.exp(p[1])
# error of tau
tau_err = np.sqrt(cov[0, 0]) / p[0]**2

# plot the fit line
x = np.linspace(bin_edges[0], bin_edges[-1], 10)
y = p[0] * x + p[1]
# y = np.log(a * np.exp(-x / tau))
ax.plot(x, 10**y, '--', label=f'$\\tau$ = {tau:.2f} $\pm$ {tau_err:.2f} $\mu$s')
ax.legend()

ax.set_ylim(1e3, 2e5)
ax.set_xlim(-0.1, 3)

df_output = pd.DataFrame({
    'voltage': np.unique(df_result.voltage_preamp1_V)[0],
    'tau': tau,
    'error': tau_err
},
index=[0]
)

# df_output.to_csv(
#     "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/afterpulse.csv",
#     mode="a",
#     header=False,
#     index=False
# )

# plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/afterpulse_decay_time.pdf")

##### double check fit

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
# this also remove null values from the peak_height_V_array
# peak_area = d2d_data.peak_area_Vns

# peak_rel_start_time = d2d_data.peak_rel_start_time_s
time_diff = df_result['time_diff']

d2d_data.get_df()

ax.set_xlabel('Time [us]')

# ax.hist(time_diff*1e6, 
#         bins=100,
#         range=[0,4],
#         ) 
# ax.axhline(1)
ax.set_ylabel('Count')

# log y
# ax.set_yscale('log')

# grid = True
ax.grid(True)

hist, bin_edges = np.histogram(time_diff*1e6, bins=100, range=[0,4])
bin_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])
hist_log = np.log(hist + 1)  # add 1 to avoid log(0)
ax.plot(bin_centers, hist_log, 'b-', label='Data')

# fit the decay trend
mask = (bin_centers >= 1) & (bin_centers <= 2.5)
# linear fit
p, cov = np.polyfit(bin_centers[mask], hist_log[mask], deg=1, cov=True)


# find tau
tau = -1 / p[0]
a = np.exp(p[1])

# plot the fit line
x = np.linspace(bin_edges[0], bin_edges[-1], 10)
y = np.log(a * np.exp(-x / tau))
ax.plot(x, y, 'r--', label=f'Fit: a*exp(-x/{tau:.2f})')
ax.legend()

# plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/area_time_nosource.pdf")

### Attempt to find the Time difference between board 0 and 1

In [ ]:
mask = d2d_data.board == 0
single_board_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

board_0_peak_time = single_board_data.peak_rel_start_time_s

mask = d2d_data.board == 1
single_board_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
board_1_peak_time = single_board_data.peak_rel_start_time_s

length = np.min([len(board_0_peak_time), len(board_1_peak_time)])
time_diff = np.abs(board_1_peak_time[:length] - board_0_peak_time[:length]) # in s

# plt.hist(time_diff, bins=50, range = [0, 1e-7])
plt.hist(time_diff, bins=50, range=[1.5e-7,0.5e-6])
plt.show()


In [ ]:
count, edge = np.histogram(time_diff, bins=50, range=[1.5e-7,0.5e-6])
edge[np.argmax(count)]  # get the bin center of the peak

#### Proxy(Event Rate) for both boards

In [ ]:
mask = (d2d_data.peak_area_PE > d2d_data.spe_position*300) & (d2d_data.board==0)
clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
cnt_0, bin_0, _ = plt.hist(clean_data.peak_start_time_s, bins=100, alpha=0.5, label='Board 0'
        #  , range = [49,51]
         )

mask = (d2d_data.peak_area_PE > d2d_data.spe_position*300) & (d2d_data.board==1)
clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
cnt_1, bin_1, _ = plt.hist(clean_data.peak_start_time_s, bins=100, alpha=0.5, label='Board 1'
        #  , range = [49,51]
         )

print("Time Diff.", bin_1[:-1][cnt_1>1][0] - bin_0[:-1][cnt_0>1][0])

plt.legend()
         
#log y
plt.yscale('log')

In [ ]:
mask = (d2d_data.peak_area_PE > d2d_data.spe_position*300) & (d2d_data.board==0)
clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
cnt_0, bin_0, _ = plt.hist(clean_data.peak_start_time_s, bins=100, alpha=0.5, label='Board 0'
         , range = [32,34]
         )

mask = (d2d_data.peak_area_PE > d2d_data.spe_position*300) & (d2d_data.board==1)
clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
cnt_1, bin_1, _ = plt.hist(clean_data.peak_start_time_s, bins=100, alpha=0.5, label='Board 1'
         , range = [32,34]
         )

print("Time Diff.", bin_1[:-1][cnt_1>1][0] - bin_0[:-1][cnt_0>1][0])

plt.legend()
         
#log y
plt.yscale('log')